# 🔥 MPS (Metal) Quick Start

This notebook verifies your PyTorch + Apple Metal (MPS) setup and runs basic GPU-accelerated operations.

In [ ]:
import torch
from metalcheck import device_info, get_device, is_mps_available
from metalcheck.utils import benchmark_matmul, system_info

## 1. Device & System Information

In [ ]:
info = device_info()
for k, v in info.items():
    print(f"{k.replace(chr(95), chr(32)).title():<20s}: {v}")

## 2. Basic Tensor Operations on Metal

In [ ]:
device = get_device()
print(f"Using device: {device}")

a = torch.randn(4, 4, device=device)
b = torch.randn(4, 4, device=device)
c = a @ b
print(f"
Matrix A:
{a}")
print(f"
Matrix B:
{b}")
print(f"
A @ B:
{c}")

## 3. MPS vs CPU Benchmark

In [ ]:
import time
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

sizes = [256, 512, 1024, 2048]
mps_times = []
cpu_times = []

for size in sizes:
    if is_mps_available():
        r = benchmark_matmul(size=size, device=torch.device("mps"), iterations=10)
        mps_times.append(r["mean_time_s"] * 1000)
    r = benchmark_matmul(size=size, device=torch.device("cpu"), iterations=10)
    cpu_times.append(r["mean_time_s"] * 1000)

print("Benchmark complete!")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = range(len(sizes))
width = 0.35

if mps_times:
    ax.bar([i - width/2 for i in x], mps_times, width, label="MPS (Metal)", color="#FF6B35")
ax.bar([i + width/2 for i in x], cpu_times, width, label="CPU", color="#4A90D9")

ax.set_xlabel("Matrix Size")
ax.set_ylabel("Time (ms)")
ax.set_title("Matrix Multiplication: MPS vs CPU")
ax.set_xticks(x)
ax.set_xticklabels([f"{s}x{s}" for s in sizes])
ax.legend()

plt.tight_layout()
plt.show()

## 4. Speedup Summary

In [ ]:
if mps_times and cpu_times:
    import pandas as pd
    df = pd.DataFrame({
        "Size": [f"{s}x{s}" for s in sizes],
        "MPS (ms)": [f"{t:.2f}" for t in mps_times],
        "CPU (ms)": [f"{t:.2f}" for t in cpu_times],
        "Speedup": [f"{c/m:.1f}x" for c, m in zip(cpu_times, mps_times)],
    })
    display(df)
else:
    print("MPS not available — ran CPU only.")